# Day 2 — Tool/Memory Validation, Multi-Hop Reasoning & Failure-Path Testing

**Module 6 · Agentic RAG Testing**

---

## What we'll cover today

| # | Topic | Why it matters |
|---|---|---|
| 1 | Why Module 5's RAGAS suite isn't enough here | Faithfulness/relevancy/precision/recall each have a specific blind spot once retrieval can loop |
| 2 | Memory validation across hops | Same boundary-value problem as Module 5's chunk-boundary bug, relocated to conversation memory |
| 3 | Hard negative: right facts, wrong combination | A failure mode per-fact faithfulness checks structurally cannot catch |
| 4 | Failure-path testing | What should happen when a hop comes up empty — the exact Klarna gap, testable |
| 5 | Extending the coverage matrix | New columns: `reasoning_chain_break`, `premature_stop`, `ungraceful_failure` |

**Estimated time:** 60 minutes
**Run order:** top to bottom. Several cells call the real agent from Day 1 -- make sure `.env` is configured in this `examples/` folder first (see Day 1's setup note).

---

> **Where we are in the course**
> Day 1 built a working, traced 2-hop loop and named 4 new failure modes that only exist in multi-step retrieval.
> Module 5 Day 2 taught boundary value analysis on chunk size; Module 5 Day 3 taught hard negatives for faithfulness.
> Today both come back, pointed at the planner's memory and its reasoning across hops instead of a single retrieval.

---
## Why Day 1's agent needs more than Module 5's test suite

Point Module 5's 4 RAGAS metrics at `agentic_rag()`'s final `response` and its accumulated `retrieved_contexts`, and three of them will happily score a *wrong* answer as fine:

- **`faithfulness`** asks "is every claim in the response backed by *some* retrieved chunk?" — it does not ask "was that the *right* chunk to base the answer on." Today's `reasoning_chain_break` hard negative cites WidgetPro 2000's real, retrieved, individually-true $50 fee — faithfulness has no way to know the question needed WidgetPro 3000's fee instead.
- **`answer_relevancy`** asks "does the response address the question asked?" — a confident, on-topic, wrong answer scores exactly the same as a confident, on-topic, right one.
- **`context_precision` / `context_recall`** ask about the *retrieved* chunks, not about what the generator did with them — they can see a retrieval error, but not a reasoning error layered on top of a clean retrieval.

None of this is a flaw in RAGAS. These metrics were built for a system where there's only ever one fact set to reason over — "did it combine multiple facts correctly" isn't a meaningful question when there's nothing to combine. It only becomes askable once retrieval can happen more than once, which is exactly what Day 1 added. Today builds the three checks that fill that gap:

1. **Memory validation** — did a fact from an early hop survive to the final answer, or get dropped along the way?
2. **Reasoning-combination checks** — were the *right* facts picked and combined, not just retrieved?
3. **Graceful-failure checks** — did the agent admit a gap instead of guessing, when a hop genuinely came up empty?

None of these three have a Module 5 equivalent — not because Module 5's authors missed them, but because none of them could occur in a system that only ever retrieves once.


---
## Memory validation: did the early fact survive?

A multi-hop chain passes facts from early hops forward into the final generation step. Module 4 Day 4 taught you to boundary-test a context window; this is the same boundary, relocated: **does the fact from hop 1 still make it into the final answer, or does it get dropped or corrupted by the time hop 3 generates a response?**

> **Plain English:** this is the chunk-boundary bug from Module 5 Day 2, except the "chunk" is the running memory of a multi-hop conversation instead of a document. The boundary is wherever your agent truncates history — and it can split a needed fact out exactly the same way a 500-token chunk boundary could.

In [1]:
from agent import agentic_rag


def facts_present_in_answer(answer: str, required_facts: list[str]) -> dict:
    missing = [f for f in required_facts if f not in answer]
    return {"missing": missing, "passed": len(missing) == 0}


required = ["WidgetPro 3000"]  # the hop-1 conclusion (WHICH product) that must survive into the final answer

# Memory intact: the REAL agent's actual answer, not a hand-typed example.
day1_result = await agentic_rag("What is the cancellation fee for the product that replaced WidgetPro 2000?")
answer_with_memory = day1_result.response

# Memory dropped: a hypothetical bug -- what it would look like if hop 1's product
# identity got truncated away before generation. We can't reliably force a real
# agent into this on demand (same reason golden_dataset.json's hard negatives are
# pre-baked, not live-generated), so this one stays hand-constructed.
answer_memory_dropped = "The cancellation fee is $0."

for label, answer in [("REAL AGENT (memory intact)", answer_with_memory),
                       ("HYPOTHETICAL (memory dropped)", answer_memory_dropped)]:
    result = facts_present_in_answer(answer, required)
    print(f"[{label}] -> passed={result['passed']}  missing={result['missing']}")
    print(f"    answer: {answer!r}")

print()
print(f"day1_result.hit_max_hops = {day1_result.hit_max_hops}  (num_hops={day1_result.num_hops})")
print("False here means the planner itself confirmed it had enough -- worth checking whenever")
print("a memory-validation case unexpectedly fails, since a True would explain a lot on its own.")


[REAL AGENT (memory intact)] -> passed=False  missing=['WidgetPro 3000']
    answer: 'The cancellation fee for the product that replaced WidgetPro 2000 is $0.'
[HYPOTHETICAL (memory dropped)] -> passed=False  missing=['WidgetPro 3000']
    answer: 'The cancellation fee is $0.'

day1_result.hit_max_hops = False  (num_hops=2)
False here means the planner itself confirmed it had enough -- worth checking whenever
a memory-validation case unexpectedly fails, since a True would explain a lot on its own.


---
## Hard negative — right facts, wrong combination (`reasoning_chain_break`)

This is the failure mode Module 5's single-hop metrics structurally cannot catch, because both retrieved facts are individually faithful to their source — the bug is in how they were *combined*.

In [2]:
# The REAL agent already answered this question in the cell above -- reuse it as
# the "correct" case instead of hand-typing one.
correct_answer = answer_with_memory

# Hard negative: the agent answers with the OLD product's fee instead of the new one's --
# every individual fact is true and grounded; the COMBINATION is wrong. Same shape as
# golden_dataset.json's multi-hop-widgetpro-01-hardneg (pre-baked, same reason as above).
reasoning_break_answer = "The cancellation fee for the product that replaced WidgetPro 2000 is $50."


def check_correct_product_fee(answer: str) -> bool:
    # The question asks about the REPLACEMENT product -- the answer must cite WidgetPro 3000's fee, not 2000's.
    return "$0" in answer and "WidgetPro 3000" in answer


for label, answer in [("HARD NEGATIVE (should fail)", reasoning_break_answer),
                       ("REAL AGENT (should pass)", correct_answer)]:
    print(f"[{label}] -> passed={check_correct_product_fee(answer)}")
    print(f"    answer: {answer!r}")

print()
print("A faithfulness check alone would PASS the hard negative above -- '$50' really is in the")
print("retrieved context. This is exactly why agentic RAG needs reasoning-level checks on top of")
print("Module 5's faithfulness/groundedness checks, not instead of them.")
print()
print("If REAL AGENT came back passed=False, that's not a broken notebook -- it means the live")
print('model phrased its answer differently than this exact-substring check expects (e.g. "free"')
print('instead of "$0"). That is a real, useful finding about the check\'s fragility, not a bug')
print("to paper over -- see Part B of this day's exercise for building a more robust version.")


[HARD NEGATIVE (should fail)] -> passed=False
    answer: 'The cancellation fee for the product that replaced WidgetPro 2000 is $50.'
[REAL AGENT (should pass)] -> passed=False
    answer: 'The cancellation fee for the product that replaced WidgetPro 2000 is $0.'

A faithfulness check alone would PASS the hard negative above -- '$50' really is in the
retrieved context. This is exactly why agentic RAG needs reasoning-level checks on top of
Module 5's faithfulness/groundedness checks, not instead of them.

If REAL AGENT came back passed=False, that's not a broken notebook -- it means the live
model phrased its answer differently than this exact-substring check expects (e.g. "free"
instead of "$0"). That is a real, useful finding about the check's fragility, not a bug
to paper over -- see Part B of this day's exercise for building a more robust version.


---
## Failure-path testing: the hop that comes up empty

What should the agent do when a hop finds nothing relevant?

- **Graceful** — admits the gap: *"I found that WidgetPro 3000 replaced WidgetPro 2000, but I don't have its cancellation fee on file."*
- **Ungraceful** — confidently invents a number to fill the gap.

This hard negative is aimed squarely at the failure pattern behind Klarna's "complex cases dropped in quality" from Day 1 — an agent that can't find the next fact should say so, not fabricate one to keep the chain moving.

In [3]:
def empty_hop_response_is_graceful(response: str) -> bool:
    hedge_phrases = ["don't have", "doesn't have", "couldn't find", "no information",
                      "not available", "not in the knowledge base", "unable to find"]
    return any(phrase in response.lower() for phrase in hedge_phrases)


# The REAL agent, asked a question the corpus genuinely has no answer for
# (no fact about TurboMax Pro's cancellation fee exists anywhere in CORPUS).
missing_fact_result = await agentic_rag("What is the cancellation fee for TurboMax Pro?", verbose=True)
real_agent_answer = missing_fact_result.response

# Ungraceful hard negative: a fabricated number, same shape as
# golden_dataset.json's graceful-failure-01-hardneg.
ungraceful = "The cancellation fee for TurboMax Pro is $25."

for label, answer in [("REAL AGENT", real_agent_answer), ("HARD NEGATIVE (fabricated)", ungraceful)]:
    print(f"[{label}] -> graceful={empty_hop_response_is_graceful(answer)}")
    print(f"    answer: {answer!r}")

print()
print(f"missing_fact_result.hit_max_hops = {missing_fact_result.hit_max_hops}")
print("This SHOULD be True -- the corpus has no answer, so the planner should never confirm")
print("enough_info. If it's False, that's worth investigating on its own: it means the planner")
print("thought it had enough to answer a question the corpus can't actually answer.")
print()
print("If REAL AGENT came back graceful=False, that IS the finding this check exists to surface --")
print("it means the generator's prompt needs a stronger instruction to hedge instead of guessing,")
print("not that the notebook is broken. 'ungraceful' is the more dangerous of the two regardless,")
print("precisely because it LOOKS like a normal, confident answer.")


[hop 1] query='What is the cancellation fee for TurboMax Pro?'
[hop 1] retrieved: ["TurboMax Pro's annual subscription costs $299, unchanged from TurboMax 5's price.", "WidgetPro 3000's cancellation fee is $0 -- it can be canceled anytime at no charge."]
[hop 1] planner.enough_info=False  reasoning="The question asks specifically for the cancellation fee of TurboMax Pro, but the retrieved facts do not mention that product's cancellation policy. The only cancellation fee listed is for WidgetPro 3000, which is unrelated. The fact about TurboMax Pro's annual subscription price does not provide cancellation fee information."
[hop 1] next_query='TurboMax Pro cancellation fee policy'

[hop 2] query='TurboMax Pro cancellation fee policy'
[hop 2] retrieved: ["TurboMax Pro's annual subscription costs $299, unchanged from TurboMax 5's price.", 'TurboMax 5 was renamed to TurboMax Pro in 2024 after a rebranding update.']
[hop 2] planner.enough_info=False  reasoning="The user asked specifically abo

---
## Extending the coverage matrix

Module 5 Day 2 added retrieval columns to Module 4 Day 4's matrix. Today adds the agentic ones -- and they map directly onto this notebook's opening argument: `reasoning_chain_break` is the check for the failure faithfulness can't see, `ungraceful_failure` is the check for the failure no RAGAS metric asks about at all.


In [4]:
import json
from collections import Counter

with open("golden_dataset.json") as f:
    golden_cases = json.load(f)

categories    = sorted({c["category"] for c in golden_cases})
failure_modes = sorted({c["failure_mode"] for c in golden_cases})
counts        = Counter((c["category"], c["failure_mode"]) for c in golden_cases)

header = " " * 16 + "".join(f"{fm:<24}" for fm in failure_modes)
print(header)
for cat in categories:
    row = f"{cat:<16}" + "".join(f"{counts[(cat, fm)]:<24}" for fm in failure_modes)
    print(row)

zero_cells = [(cat, fm) for cat in categories for fm in failure_modes if counts[(cat, fm)] == 0]
print()
if zero_cells:
    print("Still-empty cells (not necessarily a problem -- just visible now):")
    for cat, fm in zero_cells:
        print(f"- {cat} x {fm}")
else:
    print("No empty cells for these categories x failure modes -- Day 1's named premature_stop gap")
    print("is filled (see multi-hop-turbomax-01 in golden_dataset.json). single_hop_qa correctly")
    print("has no agentic-failure rows -- those columns only apply once a question needs multiple hops.")


                hallucination           premature_stop          reasoning_chain_break   ungraceful_failure      
multi_hop_qa    2                       2                       2                       2                       
single_hop_qa   2                       0                       0                       0                       

Still-empty cells (not necessarily a problem -- just visible now):
- single_hop_qa x premature_stop
- single_hop_qa x reasoning_chain_break
- single_hop_qa x ungraceful_failure


---
## Try It Yourself

1. Write a third memory-validation case where the final answer reflects hop 1's fact but **drops hop 2's entirely**. Does `facts_present_in_answer()` catch it the same way it caught the fully-dropped case above?
2. Add a `query_drift` row to `golden_dataset.json` -- a case where the reformulated query in hop 2 wanders away from the original question's intent (`query_drift` is named in Day 1's failure-mode table but has no row yet). What would its `retrieved_contexts` need to look like for a hard negative to actually demonstrate the drift, rather than just a wrong answer?
3. Write your own "right facts, wrong combination" hard negative in a domain other than product fees (e.g. dates, locations, prices) and add it to `golden_dataset.json` with `eval_type: "reasoning"`, `must_include`, and `must_not_include` fields (see `multi-hop-widgetpro-01` for the pattern). What made it easy or hard to construct compared to the WidgetPro example?

Exercise file: [`exercises/02_tool_memory_reasoning_exercise.md`](../exercises/02_tool_memory_reasoning_exercise.md)


---
## Summary

### What we built today
- A memory-validation check that catches facts dropped between hops — run against the REAL agent's answer, not a scripted stand-in — the conversational version of Module 5's chunk-boundary bug
- A `reasoning_chain_break` hard negative that a per-fact faithfulness check would have missed entirely
- A graceful-vs-ungraceful failure check aimed at the exact gap behind the Klarna incident
- A coverage matrix extended with `reasoning_chain_break`, `premature_stop`, and `ungraceful_failure`

### The thread through this whole module
Every check built across these two days reused a Module 4/5 technique and pointed it one level up: equivalence partitioning -> hop count, boundary value analysis -> conversation memory, hard negatives -> reasoning combination and graceful failure, coverage matrix -> agentic failure modes. Same mindset, new surface — same habit this course keeps building.

**Next:** Module 7 — AI Agents Testing with DeepEval, where these hand-rolled checks become formal, reusable metrics (task completion, tool correctness, argument correctness) and the agent gains real tool-calling, not just retrieval.

---